In [5]:
import torch
import torchvision
import sklearn
import pandas as pd
import matplotlib as plt
import seaborn as sb
import os
import shutil
from PIL import Image
import imagehash
from sklearn.model_selection import train_test_split
from collections import defaultdict,Counter
import torch.nn as nn
from torchvision import datasets, transforms,models
import torch.optim as optim
import time
from dataset import get_data_loaders
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
import json
import numpy as np
from torch.optim import AdamW


In [6]:
def audit_dataset(base_dir="..", hamming_threshold=5):
    splits = ["train", "test", "unclean"]
    all_images = []
    class_counts = {split: Counter() for split in splits}
    unreadable_files = []
    unusual_sizes = []

    for split in splits:
        split_dir = os.path.join(base_dir, split)
        if not os.path.exists(split_dir):
            print(f"Warning: Directory {split_dir} not found. Please ensure the dataset is extracted correctly.")
            continue
            
        print(f"Scanning split: {split}...")
        for root, _, files in os.walk(split_dir):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp')):
                    img_path = os.path.join(root, file)
                    
              
                    rel_path = os.path.relpath(img_path, split_dir)
                    path_parts = rel_path.split(os.sep)
                    if len(path_parts) > 1:
                        class_name = path_parts[0]
                        class_counts[split][class_name] += 1

                    try:
                        with Image.open(img_path) as img:
                            width, height = img.size
                            
                            if width < 32 or height < 32:
                                unusual_sizes.append((img_path, (width, height)))

                            
                            img_hash = imagehash.phash(img)
                            full_rel_path = os.path.relpath(img_path, base_dir)
                            all_images.append((full_rel_path, img_hash))
                            
                    except Exception as e:
                        unreadable_files.append((img_path, str(e)))


    duplicates = []
    visited = set()
    
    for i in range(len(all_images)):
        if i in visited:
            continue
        group = [all_images[i][0]]
        for j in range(i + 1, len(all_images)):
            if j in visited:
                continue

            hamming_dist = all_images[i][1] - all_images[j][1]
            if hamming_dist <= hamming_threshold:
                group.append(all_images[j][0])
                visited.add(j)
        if len(group) > 1:
            duplicates.append(group)

   
    print("\n" + "="*60)
    print("--- DATASET AUDIT REPORT (WITH HAMMING DISTANCE) ---")
    print("="*60)
    
    print(f"\n1. Unreadable or corrupted files: {len(unreadable_files)}")
    for path, err in unreadable_files:
        print(f"   - {path}: {err}")

    print(f"\n2. Unusually small images (< 32x32): {len(unusual_sizes)}")
    for path, size in unusual_sizes[:5]:
        print(f"   - {path} (Size: {size})")

    print("\n3. Class distribution per split:")
    for split, counts in class_counts.items():
        print(f"   [{split}]:")
        for cls, cnt in counts.items():
            print(f"     - {cls}: {cnt}")

    print(f"\n4. Similar/Duplicate groups found (Threshold <= {hamming_threshold}): {len(duplicates)}")
    for idx, group in enumerate(duplicates[:10]):
        print(f"   Group {idx + 1}:")
        for p in group:
            print(f"     * {p}")
    if len(duplicates) > 10:
        print(f"   ... and {len(duplicates) - 10} more groups.")

    print("\nDataset audit with Hamming distance completed successfully!")

if __name__ == "__main__":
    
    audit_dataset(base_dir="..")

Scanning split: train...
Scanning split: test...
Scanning split: unclean...

--- DATASET AUDIT REPORT (WITH HAMMING DISTANCE) ---

1. Unreadable or corrupted files: 0

2. Unusually small images (< 32x32): 0

3. Class distribution per split:
   [train]:
     - ambulance: 50
     - autobus: 50
     - kamyun: 50
     - kamyunet: 50
     - minibus: 50
     - savari: 50
     - taxi: 50
     - vanet: 50
   [test]:
     - ambulance: 50
     - autobus: 50
     - kamyun: 50
     - kamyunet: 50
     - minibus: 50
     - savari: 50
     - taxi: 50
     - vanet: 50
   [unclean]:
     - ambulance: 50
     - autobus: 50
     - kamyun: 50
     - kamyunet: 50
     - minibus: 50
     - neysan: 50
     - savari: 50
     - taxi: 50
     - vanet: 50

4. Similar/Duplicate groups found (Threshold <= 5): 29
   Group 1:
     * train\ambulance\199984667.jpg
     * unclean\ambulance\199984667.jpg
   Group 2:
     * train\ambulance\206808986.jpg
     * unclean\ambulance\206808986.jpg
   Group 3:
     * train\amb

# cleaning dataset

In [7]:
def clean_and_split_dataset(base_dir=".", output_dir="dataset_cleaned", hamming_threshold=5, test_size=0.2, random_seed=42):
    splits = ["train", "test", "unclean"]
    all_records = []
    
    print("1. Scanning and collecting image records...")
    for split in splits:
        split_dir = os.path.join(base_dir, split)
        if not os.path.exists(split_dir):
            continue
            
        for root, _, files in os.walk(split_dir):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp')):
                    img_path = os.path.join(root, file)
                    rel_path = os.path.relpath(img_path, base_dir)
                    path_parts = rel_path.split(os.sep)
                    
                    if len(path_parts) > 1:
                        class_name = path_parts[1] 
                        
                        
                        if class_name == "neysan":
                            continue
                            
                        try:
                            with Image.open(img_path) as img:
                                img_hash = imagehash.phash(img)
                                all_records.append({
                                    "path": img_path,
                                    "class": class_name,
                                    "hash": img_hash,
                                    "split": split
                                })
                        except Exception as e:
                            print(f"Skipping unreadable file {img_path}: {e}")

    print(f"Total valid images collected (excluding neysan): {len(all_records)}")

    print("2. Removing duplicates using Hamming distance...")
    drop_indices = set()
    for i in range(len(all_records)):
        if i in drop_indices:
            continue
        for j in range(i + 1, len(all_records)):
            if j in drop_indices:
                continue
            
            if all_records[i]["hash"] - all_records[j]["hash"] <= hamming_threshold:
                drop_indices.add(j)

    clean_records = [rec for idx, rec in enumerate(all_records) if idx not in drop_indices]
    print(f"Images remaining after duplicate removal: {len(clean_records)}")


    paths = [rec["path"] for rec in clean_records]
    labels = [rec["class"] for rec in clean_records]

    print("3. Creating 80/20 stratified train/validation split...")
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        paths, labels, test_size=test_size, random_state=random_seed, stratify=labels
    )

 
    def save_to_split(file_paths, target_split_name):
        for path in file_paths:
            parts = path.split(os.sep)
            class_name = parts[-2]
            file_name = parts[-1]
            
            dest_dir = os.path.join(output_dir, target_split_name, class_name)
            os.makedirs(dest_dir, exist_ok=True)
            shutil.copy(path, os.path.join(dest_dir, file_name))

 
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
        
    save_to_split(train_paths, "train")
    save_to_split(val_paths, "val")

    print(f"\nDataset successfully cleaned and split into '{output_dir}/train' and '{output_dir}/val'!")
    print(f"Training samples: {len(train_paths)}")
    print(f"Validation samples: {len(val_paths)}")
if __name__ == "__main__":
    clean_and_split_dataset(base_dir="..", output_dir="../dataset_cleaned")

1. Scanning and collecting image records...
Total valid images collected (excluding neysan): 1200
2. Removing duplicates using Hamming distance...
Images remaining after duplicate removal: 1166
3. Creating 80/20 stratified train/validation split...

Dataset successfully cleaned and split into '../dataset_cleaned/train' and '../dataset_cleaned/val'!
Training samples: 932
Validation samples: 234


## Data_loader and Transform

In [8]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

def get_data_loaders(data_dir="../dataset_cleaned", batch_size=32, img_size=224):

        
    train_transforms = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(), 
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406], 
            std=[0.229, 0.224, 0.225]
        ) 
    ])

   
    val_transforms = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406], 
            std=[0.229, 0.224, 0.225]
        )
    ])
    train_path = os.path.join(data_dir, "train")
    val_path = os.path.join(data_dir, "val")


    train_dataset = datasets.ImageFolder(root=train_path, transform=train_transforms)
    val_dataset = datasets.ImageFolder(root=val_path, transform=val_transforms)


    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=0
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=2
    )

    print(f"Classes detected: {train_dataset.classes}")
    print(f"Total training samples: {len(train_dataset)}")
    print(f"Total validation samples: {len(val_dataset)}")

    return train_loader, val_loader

train_loader, val_loader = get_data_loaders()

for images, labels in train_loader:
    print(f"Batch images shape: {images.shape}")
    print(f"Batch labels shape: {labels.shape}")
    break

Classes detected: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']
Total training samples: 932
Total validation samples: 234
Batch images shape: torch.Size([32, 3, 224, 224])
Batch labels shape: torch.Size([32])


### transformation to GPU

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Active Device: {device}")

if device.type == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU Model detected: {gpu_name}")
    print(f"Total GPU Memory: {round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)} GB")
else:
    print("No NVIDIA GPU detected by PyTorch. It is running on CPU.")

Active Device: cuda
GPU Model detected: NVIDIA GeForce GTX 1650 with Max-Q Design
Total GPU Memory: 4.0 GB


In [10]:
!nvidia-smi


Sat Sep 26 22:25:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 616.92                 KMD Version: 616.92        CUDA UMD Version: 13.4     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   50C    P8              2W /   35W |     515MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Model prepration

In [11]:

def create_model(num_classes=8):
    
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    
    for param in model.parameters():
        param.requires_grad = False
        
   
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    
    return model

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    
    my_model = create_model(num_classes=8)
    my_model = my_model.to(device)
    
    
    print("--- Active layers for training ---")
    active_layers = 0
    for name, param in my_model.named_parameters():
        if param.requires_grad:
            print(f"Training: {name}")
            active_layers += 1
            
    if active_layers == 2:
        print("\n✅ Model successfully frozen, only the last layer is ready for training.")

Using device: cuda
--- Active layers for training ---
Training: fc.weight
Training: fc.bias

✅ Model successfully frozen, only the last layer is ready for training.


# Training loop

In [12]:
def train_model(num_epochs=5):
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f" Training running on: {device}")
    if device.type == 'cuda':
        print(f" GPU in use: {torch.cuda.get_device_name(0)}")

    
    train_loader, val_loader = get_data_loaders(data_dir="../dataset_cleaned")
    model = create_model(num_classes=8).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.fc.parameters(), lr=0.001)

   
    best_metrics = {
        'epoch': 0,
        'train_acc': 0.0,
        'train_loss': 0.0,
        'val_acc': 0.0,
        'val_loss': 0.0
    }

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 25)
        start_time = time.time()

    
        model.train()
        train_loss = 0.0
        train_corrects = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            train_corrects += torch.sum(preds == labels.data)

        epoch_train_loss = train_loss / len(train_loader.dataset)
        epoch_train_acc = train_corrects.double() / len(train_loader.dataset)

        
        model.eval()
        val_loss = 0.0
        val_corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)

                val_loss += loss.item() * inputs.size(0)
                val_corrects += torch.sum(preds == labels.data)

        epoch_val_loss = val_loss / len(val_loader.dataset)
        epoch_val_acc = val_corrects.double() / len(val_loader.dataset)

       
        print(f"Train -> Loss: {epoch_train_loss:.4f} | Acc: {epoch_train_acc:.4f}")
        print(f"Val   -> Loss: {epoch_val_loss:.4f} | Acc: {epoch_val_acc:.4f}")

        
        if epoch_val_acc > best_metrics['val_acc']:
            best_metrics['epoch'] = epoch + 1
            best_metrics['val_acc'] = epoch_val_acc
            best_metrics['val_loss'] = epoch_val_loss
            best_metrics['train_acc'] = epoch_train_acc
            best_metrics['train_loss'] = epoch_train_loss
            
            torch.save(model.state_dict(), 'best_model.pth')
            print(" Best model weights saved successfully!")
            
        print(f"Duration: {time.time() - start_time:.0f}s")

    
    print(f"\n Training finished! Best metrics were achieved in Epoch {best_metrics['epoch']}:")
    print(f" Best Val   -> Loss: {best_metrics['val_loss']:.4f} | Acc: {best_metrics['val_acc']:.4f}")
    print(f" Same Epoch Train -> Loss: {best_metrics['train_loss']:.4f} | Acc: {best_metrics['train_acc']:.4f}")

if __name__ == "__main__":
    train_model(num_epochs=5)

 Training running on: cuda
 GPU in use: NVIDIA GeForce GTX 1650 with Max-Q Design
Classes detected: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']
Total training samples: 932
Total validation samples: 234

Epoch 1/5
-------------------------
Train -> Loss: 1.8023 | Acc: 0.3723
Val   -> Loss: 1.3831 | Acc: 0.6282
 Best model weights saved successfully!
Duration: 35s

Epoch 2/5
-------------------------
Train -> Loss: 1.2008 | Acc: 0.6964
Val   -> Loss: 1.0552 | Acc: 0.7265
 Best model weights saved successfully!
Duration: 19s

Epoch 3/5
-------------------------
Train -> Loss: 0.9365 | Acc: 0.7822
Val   -> Loss: 0.9160 | Acc: 0.7393
 Best model weights saved successfully!
Duration: 11s

Epoch 4/5
-------------------------
Train -> Loss: 0.7882 | Acc: 0.7994
Val   -> Loss: 0.7915 | Acc: 0.7650
 Best model weights saved successfully!
Duration: 10s

Epoch 5/5
-------------------------
Train -> Loss: 0.6884 | Acc: 0.8348
Val   -> Loss: 0.7095 | Acc: 0.7

In [13]:
import sys
print(sys.executable)

c:\Users\Elahe\anaconda3\envs\vehicle_env\python.exe


# Evaluation

In [14]:
def load_trained_model(checkpoint_path="../best_model.pth", num_classes=8, device="cuda"):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)
        
    model.to(device)
    model.eval()
    return model

def predict_single_image(model, image_path, class_names, transform, device="cuda", threshold=0.70):
    from PIL import Image
    image = Image.open(image_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]
        conf, pred_idx = torch.max(probabilities, dim=0)
        
    pred_class = class_names[pred_idx.item()]
    confidence = conf.item()
    
    prob_dict = {class_names[i]: round(probabilities[i].item(), 4) for i in range(len(class_names))}
    
    result = {
        "predicted_class": pred_class,
        "confidence": round(confidence, 4),
        "probabilities": prob_dict,
        "needs_review": confidence < threshold
    }
    return result

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Evaluating using device: {device}")
    
    # تنظیمات ترنسفرم برای ارزیابی
    eval_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    test_dataset = datasets.ImageFolder("../test", transform=eval_transform)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    
   
    model = load_trained_model("best_model.pth", num_classes=len(test_dataset.classes), device=device)
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            
    acc = np.mean(np.array(all_preds) == np.array(all_targets))
    macro_p = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    macro_r = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    
    print("\n--- Evaluation Results on Test Set ---")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro Precision: {macro_p:.4f}")
    print(f"Macro Recall: {macro_r:.4f}")
    print(f"Macro F1-Score: {macro_f1:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(all_targets, all_preds, target_names=test_dataset.classes))

Evaluating using device: cuda


C:\Users\Elahe\AppData\Local\Temp\ipykernel_6880\4143182973.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=device)


--- Evaluation Results on Test Set ---
Accuracy: 0.8325
Macro Precision: 0.8423
Macro Recall: 0.8325
Macro F1-Score: 0.8304

Classification Report:
              precision    recall  f1-score   support

   ambulance       0.88      0.90      0.89        50
     autobus       0.82      0.82      0.82        50
      kamyun       0.62      0.82      0.71        50
    kamyunet       0.84      0.52      0.64        50
     minibus       0.92      0.88      0.90        50
      savari       0.86      0.96      0.91        50
        taxi       0.96      0.88      0.92        50
       vanet       0.85      0.88      0.86        50

    accuracy                           0.83       400
   macro avg       0.84      0.83      0.83       400
weighted avg       0.84      0.83      0.83       400



### number of parameters

In [15]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 11,180,616
Trainable Parameters: 11,180,616


# Unfreeze layer 4

# Train fine tune model (unfreeze layer 4)

In [16]:
def prepare_finetune_model(checkpoint_path="best_model.pth", num_classes=8, device="cuda"):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    
    # 2. Load our previous best feature-extraction weights
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        if "model_state_dict" in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
        else:
            model.load_state_dict(checkpoint)
        print(f"Loaded weights from {checkpoint_path}")
    else:
        print("Warning: No checkpoint found, starting from random/pretrained base.")
        
    for param in model.parameters():
        param.requires_grad = False
    for param in model.layer4.parameters():
        param.requires_grad = True
    for param in model.fc.parameters():
        param.requires_grad = True
        
    model.to(device)
    return model

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Fine-tuning using device: {device}")
    
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    eval_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    # Load dataset splits
    train_dataset = datasets.ImageFolder("../train", transform=train_transform)
    
    # Create a train/validation split (80/20 rule)
    val_size = int(0.2 * len(train_dataset))
    train_size = len(train_dataset) - val_size
    train_subset, val_subset = random_split(
        train_dataset, [train_size, val_size], 
        generator=torch.Generator().manual_seed(42)
    )
    
    # Apply evaluation transform to validation subset
    val_subset.dataset.transform = eval_transform
    
    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)
    
    # Prepare model for fine-tuning
    model = prepare_finetune_model("best_model.pth", num_classes=len(train_dataset.classes), device=device)
    
    # Setup differential learning rates for optimizer
    optimizer = AdamW([
        {"params": model.layer4.parameters(), "lr": 1e-5}, # Tiny step for layer4
        {"params": model.fc.parameters(), "lr": 1e-3}      # Larger step for classification head
    ], weight_decay=1e-4)
    
    criterion = nn.CrossEntropyLoss()
    num_epochs = 5
    best_val_acc = 0.0
    
    # Print out trainable parameters to verify layer4 and fc are open
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Parameters for Fine-Tuning: {trainable_params:,} out of {total_params:,}")
    
    print("\nStarting Fine-Tuning Training Loop...")
    for epoch in range(num_epochs):
        # --- Training Phase ---
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct_train += (preds == targets).sum().item()
            total_train += targets.size(0)
            
        train_loss = running_loss / total_train
        train_acc = correct_train / total_train
        
        # --- Validation Phase ---
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0
        
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                
                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                correct_val += (preds == targets).sum().item()
                total_val += targets.size(0)
                
        val_loss = val_loss / total_val
        val_acc = correct_val / total_val
        
        print(f"Epoch [{epoch+1}/{num_epochs}] | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        # Save best fine-tuned model checkpoint
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc
            }, "best_finetune_model.pth")
            print(f"--> Saved new best fine-tuned model with Val Acc: {val_acc:.4f}")

    print("\nFine-Tuning Complete! Best checkpoint saved as 'best_finetune_model.pth'.")

Fine-tuning using device: cuda


C:\Users\Elahe\AppData\Local\Temp\ipykernel_6880\2143147942.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=device)

Loaded weights from best_model.pth
Trainable Parameters for Fine-Tuning: 8,397,832 out of 11,180,616

Starting Fine-Tuning Training Loop...
Epoch [1/5] | Train Loss: 0.6224 | Train Acc: 0.8187 | Val Loss: 0.6304 | Val Acc: 0.8000
--> Saved new best fine-tuned model with Val Acc: 0.8000
Epoch [2/5] | Train Loss: 0.4672 | Train Acc: 0.9000 | Val Loss: 0.5831 | Val Acc: 0.7875
Epoch [3/5] | Train Loss: 0.3493 | Train Acc: 0.9469 | Val Loss: 0.5575 | Val Acc: 0.8125
--> Saved new best fine-tuned model with Val Acc: 0.8125
Epoch [4/5] | Train Loss: 0.2501 | Train Acc: 0.9750 | Val Loss: 0.5552 | Val Acc: 0.8125
Epoch [5/5] | Train Loss: 0.1977 | Train Acc: 0.9875 | Val Loss: 0.5345 | Val Acc: 0.8375
--> Saved new best fine-tuned model with Val Acc: 0.8375

Fine-Tuning Complete! Best checkpoint saved as 'best_finetune_model.pth'.


# Simple CNN Medel

In [18]:
class SimpleTrafficCNN(nn.Module):
    def __init__(self, num_classes=8):
        super(SimpleTrafficCNN, self).__init__()
        
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) 
        )
        
        self.block2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
      
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 56 * 56, 128),
            nn.ReLU(),
            nn.Dropout(0.5), 
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.classifier(x)
        return x

if __name__ == "__main__":
    model = SimpleTrafficCNN(num_classes=8)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Simple Custom CNN Total Parameters: {total_params:,}")

model.to(device)

model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for inputs, targets in val_loader: 
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())


acc = np.mean(np.array(all_preds) == np.array(all_targets))
macro_p = precision_score(all_targets, all_preds, average='macro', zero_division=0)
macro_r = recall_score(all_targets, all_preds, average='macro', zero_division=0)
macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)


print(f"\n--- Final Model Evaluation Metrics Simple CNN---")
print(f"Accuracy: {acc:.4f}")
print(f"Macro Precision: {macro_p:.4f}")
print(f"Macro Recall: {macro_r:.4f}")
print(f"Macro F1-Score: {macro_f1:.4f}")


print("\nClassification Report:")
print(classification_report(all_targets, all_preds))

Simple Custom CNN Total Parameters: 25,710,664

--- Final Model Evaluation Metrics Simple CNN---
Accuracy: 0.1750
Macro Precision: 0.1080
Macro Recall: 0.1684
Macro F1-Score: 0.1179

Classification Report:
              precision    recall  f1-score   support

           0       0.17      0.64      0.27        11
           1       0.31      0.36      0.33        11
           2       0.05      0.12      0.07         8
           3       0.00      0.00      0.00        11
           4       0.00      0.00      0.00         9
           5       0.00      0.00      0.00        11
           6       0.00      0.00      0.00        10
           7       0.33      0.22      0.27         9

    accuracy                           0.17        80
   macro avg       0.11      0.17      0.12        80
weighted avg       0.11      0.17      0.12        80



c:\Users\Elahe\anaconda3\envs\vehicle_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Elahe\anaconda3\envs\vehicle_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Elahe\anaconda3\envs\vehicle_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

# proposed CNN Model

In [ ]:
class TrafficNetProficient(nn.Module):
    def __init__(self, num_classes=8):
        super(TrafficNetProficient, self).__init__()
        
       
        self.feature_extractor = nn.Sequential(
            
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), 
            
          
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), 
            
            
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.classifier(x)
        return x

if __name__ == "__main__":
    model = TrafficNetProficient(num_classes=8)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Proficient Custom CNN Total Parameters: {total_params:,}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TrafficNetProficient(num_classes=8).to(device) 

print(f"Model successfully loaded on: {device}")
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable Parameters: {total_params:,}")


model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for inputs, targets in val_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())


acc = np.mean(np.array(all_preds) == np.array(all_targets))
macro_p = precision_score(all_targets, all_preds, average='macro', zero_division=0)
macro_r = recall_score(all_targets, all_preds, average='macro', zero_division=0)
macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)

print(f"\n--- Results for Proficient Custom CNN ---")
print(f"Accuracy: {acc:.4f}")
print(f"Macro Precision: {macro_p:.4f}")
print(f"Macro Recall: {macro_r:.4f}")
print(f"Macro F1-Score: {macro_f1:.4f}")

Proficient Custom CNN Total Parameters: 102,472
Model successfully loaded on: cuda
Trainable Parameters: 102,472

--- Results for Proficient Custom CNN ---
Accuracy: 0.1375
Macro Precision: 0.0172
Macro Recall: 0.1250
Macro F1-Score: 0.0302
